# Data Preparation
This [dataset](https://www.kaggle.com/datasets/rabieelkharoua/air-quality-and-health-impact-dataset) contains comprehensive information on the air quality and its impact on public health for 5,811 records. It includes variables such as air quality index (AQI), concentrations of various pollutants, weather conditions, and health impact metrics. The target variable is the health impact class, which categorizes the health impact based on the air quality and other related factors.

This dataset offers a comprehensive view of the relationship between air quality and public health, making it ideal for research, predictive modeling, and statistical analysis.

In [1]:
include("utils/utils.jl")

   Resolving package versions...
  No Changes to `C:\Users\alons\.julia\environments\v1.11\Project.toml`
  No Changes to `C:\Users\alons\.julia\environments\v1.11\Manifest.toml`


oneHotEncoding (generic function with 3 methods)

In [2]:
# Load the dataset from the 'dataset' folder
data = CSV.read("datasets/air_quality_health_impact_data.csv", DataFrame)

# Check the dataset
describe(data)

Row,variable,mean,min,median,max,nmissing,eltype
,Symbol,Float64,Real,Float64,Real,Int64,DataType
1,RecordID,2906.0,1,2906.0,5811,0,Int64
2,AQI,248.438,0.00581738,249.128,499.859,0,Float64
3,PM10,148.655,0.0158481,147.635,299.902,0,Float64
4,PM2_5,100.224,0.0315489,100.506,199.985,0,Float64
5,NO2,102.293,0.00962478,102.988,199.98,0,Float64
6,SO2,49.4568,0.0110232,49.5302,99.9696,0,Float64
7,O3,149.312,0.001661,149.56,299.937,0,Float64
8,Temperature,14.9755,-9.991,14.9424,39.9634,0,Float64
9,Humidity,54.7769,10.0015,54.5439,99.9975,0,Float64


In [3]:
input_data = Matrix(data[!, 1:13]);
output_data = Int.(data[!, 15]);

@assert input_data isa Matrix
@assert output_data isa Vector{Int64}

In [4]:
output_data = oneHotEncoding(vec(output_data))

5811×5 BitMatrix:
 1  0  0  0  0
 1  0  0  0  0
 1  0  0  0  0
 1  0  0  0  0
 1  0  0  0  0
 0  1  0  0  0
 0  1  0  0  0
 1  0  0  0  0
 1  0  0  0  0
 1  0  0  0  0
 ⋮           
 0  0  1  0  0
 0  0  1  0  0
 0  0  1  0  0
 0  1  0  0  0
 0  0  0  0  1
 0  0  1  0  0
 0  1  0  0  0
 0  0  0  0  1
 1  0  0  0  0

In [5]:
input_data

5811×13 Matrix{Float64}:
    1.0  187.27    295.853     13.0386   …  84.4243   6.13776   7.0  5.0  1.0
    2.0  475.357   246.255      9.9845      46.8514   4.52142  10.0  2.0  0.0
    3.0  365.997    84.4432    23.1113      17.807   11.1574   13.0  3.0  0.0
    4.0  299.329    21.0206    14.2734      99.4734  15.3025    8.0  8.0  1.0
    5.0   78.0093   16.9877   152.112       24.9068  14.5347    9.0  0.0  1.0
    6.0   77.9973   36.1134    97.1132   …  32.6359   4.67513  13.0  5.0  2.0
    7.0   29.0418  174.231     68.5784      24.6793   6.61005  10.0  2.0  2.0
    8.0  433.088   278.629     83.6738      40.3732  17.3766   11.0  8.0  1.0
    9.0  300.558   149.023    185.789       36.0352  14.4649    8.0  6.0  4.0
   10.0  354.036   252.884    182.15        64.0992  14.2539   13.0  5.0  1.0
    ⋮                                    ⋱                      ⋮         
 5803.0  135.859   290.946     74.2386      60.6339   1.74865   8.0  7.0  0.0
 5804.0  407.978     7.50123    4.51233   

In [6]:
function normalizeData(train_inputs::AbstractArray{<:Real, 2},
    test_inputs::AbstractArray{<:Real, 2},
    normalizationType::Symbol)
    
    if normalizationType == :MinMax
        parameters = calculateMinMaxNormalizationParameters(train_input)
        # normalize the train using the previous parameters
        new_train_input = normalizeMinMax(train_input, parameters)
        # normalize the test using the  train parameters
        new_test_input = normalizeMinMax(test_input, parameters)
    elseif normalizationType == :ZeroMean
        parameters = calculateZeroMeanNormalizationParameters(train_input)
        # normalize the train using the previous parameters
        new_train_input = normalizeZeroMean(train_input, parameters)
        # normalize the test using the  train parameters
        new_test_input = normalizeZeroMean(test_input, parameters)
    end

    return (new_train_input, new_test_input)
end;

In [7]:
# Split in train and test
(tr_idx, test_idx) = holdOut(size(input_data, 1), 0.2)


train_input = input_data[tr_idx,:]
train_output = output_data[tr_idx, :]
test_input = input_data[test_idx,:]
test_output = output_data[test_idx, :]

train_output = collect(train_output)
test_output = collect(test_output)

norm_train_input_minmax, norm_test_input_minmax = normalizeData(train_input, test_input, :MinMax) 
norm_train_input_zeromean, norm_test_input_zeromean = normalizeData(train_input, test_input, :ZeroMean)

println("MinMax train input", norm_train_input_minmax[:,10])
println("MinMax test input", norm_test_input_minmax[:,10])

println("ZeroMean train input", norm_train_input_zeromean[:,10])
println("ZeroMean test input", norm_test_input_zeromean[:,10])

MinMax train input[0.3622228012220535, 0.025090310728493007, 0.8439824174785145, 0.11954303143311486, 0.13183251572525742, 0.16566115823632802, 0.46933702972520763, 0.760065129196753, 0.2555776843137259, 0.35012006594277517, 0.23800053099688065, 0.5140787887498467, 0.1015989026444768, 0.7920166224244257, 0.5098127151345374, 0.29668258873688735, 0.9627860172222811, 0.634441692046907, 0.8434233402562319, 0.700169632937471, 0.7600337846217682, 0.07565286967111776, 0.7527543127726548, 0.4593897433938246, 0.39852201862030373, 0.1771642436824931, 0.42227077456845796, 0.8133392112876858, 0.6069814521134032, 0.7081157624986011, 0.4088566957826216, 0.5190350443633003, 0.15203709389041037, 0.5878847221415663, 0.24592876014394374, 0.35933047169191656, 0.4600153385545166, 0.8662645836024753, 0.5002468169903859, 0.10574572823630862, 0.5198923837412823, 0.507445665947729, 0.2717150213134811, 0.43238905710520364, 0.9434936637208564, 0.8423059130779762, 0.7777329512514694, 0.2778814141728169, 0.405420

In [9]:
println("DIMENSIONS:")
println("train_input:",size(norm_train_input_minmax))
println("train_output:",size(train_output))
println("test_input:",size(norm_train_input_zeromean))
println("test_output:",size(test_output))

DIMENSIONS:
train_input:(4649, 13)
train_output:(4649, 5)
test_input:(4649, 13)
test_output:(1162, 5)


In [ ]:
kFoldIndices = crossvalidation(size(output_data,1), 10)
#Model type for SVM
estimators = [:SVM, :DecisionTree, :KNN, :ANN, :ANN]

# Model hyperparameters specific
modelsHyperParameters = [Dict(
    "kernel"=> "rbf",
    "degree"=> 3,
    "gamma"=> 0.0,
    "C" => 1.0 ),
    
    Dict(
        "max_depth" => 5,
        "random_state" => 42
    ),
    
    Dict(
        "k" => 5 
    ),
    
    Dict(
        "topology" => (100, 50),        
        "maxEpochs" => 200,             
        "learningRate" => 0.001,         
        "validation_fraction" => 0.1
    ),
     
    Dict(
        "topology" => (100, 50),        
        "maxEpochs" => 200,             
        "learningRate" => 0.001         
    )  
]

output_data = collect(reshape(output_data,:,1));
dataset = (input_normal, output_data);

In [ ]:
trainClassEnsemble( estimators, modelsHyperParameters, dataset, kFoldIndices)